In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = Path("../data/raw")

print("DeliveryRisk Engine — Historical Intelligence")

DeliveryRisk Engine — Historical Intelligence


In [2]:
orders = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv"
)

customers = pd.read_csv(
    DATA_DIR / "olist_customers_dataset.csv"
)

order_items = pd.read_csv(
    DATA_DIR / "olist_order_items_dataset.csv"
)

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Order items:", order_items.shape)

Orders: (99441, 8)
Customers: (99441, 5)
Order items: (112650, 7)


In [3]:
timestamp_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in timestamp_columns:
    orders[col] = pd.to_datetime(orders[col])

customer_lookup = customers[
    [
        "customer_id",
        "customer_unique_id"
    ]
].copy()

delivered = (
    orders[
        orders["order_status"] == "delivered"
    ]
    .copy()
    .merge(
        customer_lookup,
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
)

delivered["delivery_delay_days"] = (
    delivered["order_delivered_customer_date"]
    - delivered["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

delivered["late_delivery"] = (
    delivered["delivery_delay_days"] > 0
).astype(int)

delivered = delivered.dropna(
    subset=["order_delivered_customer_date"]
)

print("Delivered rows:", len(delivered))
print(
    "Late deliveries:",
    delivered["late_delivery"].sum()
)

Delivered rows: 96470
Late deliveries: 7826


In [4]:
seller_orders = (
    order_items[
        [
            "order_id",
            "seller_id"
        ]
    ]
    .drop_duplicates()
    .merge(
        delivered[
            [
                "order_id",
                "order_purchase_timestamp",
                "order_delivered_customer_date",
                "late_delivery",
                "delivery_delay_days"
            ]
        ],
        on="order_id",
        how="inner",
        validate="many_to_one"
    )
)

print("Seller-order records:", len(seller_orders))

Seller-order records: 97811


In [5]:
seller_events = (
    seller_orders[
        [
            "seller_id",
            "order_id",
            "order_delivered_customer_date",
            "late_delivery",
            "delivery_delay_days"
        ]
    ]
    .dropna(
        subset=["order_delivered_customer_date"]
    )
    .sort_values(
        [
            "order_delivered_customer_date",
            "seller_id"
        ]
    )
    .reset_index(drop=True)
)

seller_events["completed_orders_so_far"] = (
    seller_events
    .groupby("seller_id")
    .cumcount() + 1
)

seller_events["completed_lates_so_far"] = (
    seller_events
    .groupby("seller_id")["late_delivery"]
    .cumsum()
)

seller_events["completed_delay_sum_so_far"] = (
    seller_events
    .groupby("seller_id")["delivery_delay_days"]
    .cumsum()
)

In [6]:
current_seller_orders = (
    seller_orders[
        [
            "order_id",
            "seller_id",
            "order_purchase_timestamp"
        ]
    ]
    .sort_values(
        [
            "order_purchase_timestamp",
            "seller_id"
        ]
    )
    .reset_index(drop=True)
)

seller_history = pd.merge_asof(
    current_seller_orders,
    seller_events[
        [
            "seller_id",
            "order_delivered_customer_date",
            "completed_orders_so_far",
            "completed_lates_so_far",
            "completed_delay_sum_so_far"
        ]
    ],
    left_on="order_purchase_timestamp",
    right_on="order_delivered_customer_date",
    by="seller_id",
    direction="backward",
    allow_exact_matches=False
)

In [7]:
seller_history["seller_previous_orders"] = (
    seller_history["completed_orders_so_far"]
    .fillna(0)
)

seller_history["seller_previous_lates"] = (
    seller_history["completed_lates_so_far"]
    .fillna(0)
)

seller_history["seller_history_available"] = (
    seller_history["seller_previous_orders"] > 0
).astype(int)

seller_history["seller_historical_late_rate"] = (
    seller_history["completed_lates_so_far"]
    / seller_history["completed_orders_so_far"]
)

seller_history["seller_historical_avg_delay"] = (
    seller_history["completed_delay_sum_so_far"]
    / seller_history["completed_orders_so_far"]
)

seller_history_features = (
    seller_history
    .groupby("order_id")
    .agg(
        seller_previous_orders=(
            "seller_previous_orders",
            "mean"
        ),
        seller_historical_late_rate=(
            "seller_historical_late_rate",
            "mean"
        ),
        seller_historical_avg_delay=(
            "seller_historical_avg_delay",
            "mean"
        ),
        seller_history_available=(
            "seller_history_available",
            "max"
        )
    )
    .reset_index()
)

seller_history_features.head()

,order_id,seller_previous_orders,seller_historical_late_rate,seller_historical_avg_delay,seller_history_available
0,00010242fe8c5a6d1ba2dd792cb16214,74.0,0.013514,-12.072272,1
1,00018f77f2f0320c557190d7a144bdd3,0.0,NaN,NaN,0
2,000229ec398224ef6ca0657da4fc703e,4.0,0.000000,-13.860472,1
3,00024acbcdf0a6daa1e931b038114c75,10.0,0.100000,-13.687788,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,2.0,0.000000,-33.279583,1


In [8]:
seller_events = (
    seller_orders[
        [
            "seller_id",
            "order_id",
            "order_delivered_customer_date",
            "late_delivery",
            "delivery_delay_days"
        ]
    ]
    .dropna(subset=["order_delivered_customer_date"])
    .sort_values(
        ["order_delivered_customer_date", "seller_id"]
    )
    .reset_index(drop=True)
)

seller_events["completed_orders_so_far"] = (
    seller_events
    .groupby("seller_id")
    .cumcount() + 1
)

seller_events["completed_lates_so_far"] = (
    seller_events
    .groupby("seller_id")["late_delivery"]
    .cumsum()
)

seller_events["completed_delay_sum_so_far"] = (
    seller_events
    .groupby("seller_id")["delivery_delay_days"]
    .cumsum()
)

print("Seller events:", len(seller_events))
print(
    "Missing delivery timestamps:",
    seller_events["order_delivered_customer_date"].isna().sum()
)

Seller events: 97811
Missing delivery timestamps: 0


In [9]:
current_seller_orders = (
    seller_orders[
        [
            "order_id",
            "seller_id",
            "order_purchase_timestamp"
        ]
    ]
    .sort_values(
        ["order_purchase_timestamp", "seller_id"]
    )
    .reset_index(drop=True)
)

seller_history = pd.merge_asof(
    current_seller_orders,
    seller_events[
        [
            "seller_id",
            "order_delivered_customer_date",
            "completed_orders_so_far",
            "completed_lates_so_far",
            "completed_delay_sum_so_far"
        ]
    ].sort_values(
        ["order_delivered_customer_date", "seller_id"]
    ),
    left_on="order_purchase_timestamp",
    right_on="order_delivered_customer_date",
    by="seller_id",
    direction="backward",
    allow_exact_matches=False
)

print("History rows:", len(seller_history))
print("Unique orders:", seller_history["order_id"].nunique())

History rows: 97811
Unique orders: 96470


In [10]:
seller_history[
    [
        "seller_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "completed_orders_so_far",
        "completed_lates_so_far"
    ]
].head(20)

,seller_id,order_purchase_timestamp,order_delivered_customer_date,completed_orders_so_far,completed_lates_so_far
0,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-15 12:16:38,NaT,NaN,NaN
1,522620dcb18a6b31cd7bdf73665113a9,2016-10-03 09:44:50,NaT,NaN,NaN
2,f09b760d23495ac9a7e00d29b769007c,2016-10-03 16:56:50,NaT,NaN,NaN
3,45d33f715e24d15a6ccf5c17b3a23e3c,2016-10-03 21:13:36,NaT,NaN,NaN
4,cca3071e3e9bb7d12640c9fbe2301306,2016-10-03 22:06:03,NaT,NaN,NaN
5,b499c00f28f4b7069ff6550af8c1348a,2016-10-03 22:31:31,NaT,NaN,NaN
6,817f85dbb65aa3e70831d90fe75cdf89,2016-10-03 22:44:10,NaT,NaN,NaN
7,cca3071e3e9bb7d12640c9fbe2301306,2016-10-03 22:51:30,NaT,NaN,NaN
8,3481aa57cd91f9f9d3fa1fa12d9a3bf7,2016-10-04 09:06:10,NaT,NaN,NaN
9,4b1eaadf791bdbbad8c4a35b65236d52,2016-10-04 09:16:33,NaT,NaN,NaN


In [11]:
seller_history["seller_previous_orders"] = (
    seller_history["completed_orders_so_far"]
    .fillna(0)
)

seller_history["seller_previous_lates"] = (
    seller_history["completed_lates_so_far"]
    .fillna(0)
)

seller_history["seller_history_available"] = (
    seller_history["seller_previous_orders"] > 0
).astype(int)

seller_history["seller_historical_late_rate"] = (
    seller_history["completed_lates_so_far"]
    / seller_history["completed_orders_so_far"]
)

seller_history["seller_historical_avg_delay"] = (
    seller_history["completed_delay_sum_so_far"]
    / seller_history["completed_orders_so_far"]
)

seller_history_features = (
    seller_history
    .groupby("order_id")
    .agg(
        seller_previous_orders=(
            "seller_previous_orders",
            "mean"
        ),
        seller_historical_late_rate=(
            "seller_historical_late_rate",
            "mean"
        ),
        seller_historical_avg_delay=(
            "seller_historical_avg_delay",
            "mean"
        ),
        seller_history_available=(
            "seller_history_available",
            "max"
        )
    )
    .reset_index()
)

print("Rows:", len(seller_history_features))
print(
    "Unique orders:",
    seller_history_features["order_id"].nunique()
)

Rows: 96470
Unique orders: 96470


In [12]:
seller_history_features.head(10)

,order_id,seller_previous_orders,seller_historical_late_rate,seller_historical_avg_delay,seller_history_available
0,00010242fe8c5a6d1ba2dd792cb16214,74.0,0.013514,-12.072272,1
1,00018f77f2f0320c557190d7a144bdd3,0.0,NaN,NaN,0
2,000229ec398224ef6ca0657da4fc703e,4.0,0.000000,-13.860472,1
3,00024acbcdf0a6daa1e931b038114c75,10.0,0.100000,-13.687788,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,2.0,0.000000,-33.279583,1
5,00048cc3ae777c65dbb7d2a0634bc1ea,0.0,NaN,NaN,0
6,00054e8431b9d7675808bcb819fb4a32,59.0,0.050847,-9.782351,1
7,000576fe39319847cbb9d288c5617fa6,0.0,NaN,NaN,0
8,0005a1a1728c9d785b8e2b08b904576c,110.0,0.100000,-9.951381,1
9,0005f50442cb953dcd1d21e1fb923495,57.0,0.157895,-7.550524,1


In [13]:
print(
    "Orders with seller history:",
    seller_history_features["seller_history_available"].sum()
)

print(
    "Orders without seller history:",
    (seller_history_features["seller_history_available"] == 0).sum()
)

Orders with seller history: 91296
Orders without seller history: 5174


In [14]:
customer_orders = delivered[
    [
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "late_delivery",
        "delivery_delay_days"
    ]
].copy()

customer_orders = customer_orders.sort_values(
    ["order_purchase_timestamp", "customer_unique_id"]
).reset_index(drop=True)

print("Customer-order records:", len(customer_orders))
print(
    "Unique customers:",
    customer_orders["customer_unique_id"].nunique()
)

Customer-order records: 96470
Unique customers: 93350


In [15]:
customer_events = (
    customer_orders[
        [
            "customer_unique_id",
            "order_id",
            "order_delivered_customer_date",
            "late_delivery",
            "delivery_delay_days"
        ]
    ]
    .dropna(subset=["order_delivered_customer_date"])
    .sort_values(
        ["order_delivered_customer_date", "customer_unique_id"]
    )
    .reset_index(drop=True)
)

customer_events["completed_orders_so_far"] = (
    customer_events
    .groupby("customer_unique_id")
    .cumcount() + 1
)

customer_events["completed_lates_so_far"] = (
    customer_events
    .groupby("customer_unique_id")["late_delivery"]
    .cumsum()
)

customer_events["completed_delay_sum_so_far"] = (
    customer_events
    .groupby("customer_unique_id")["delivery_delay_days"]
    .cumsum()
)

print("Customer events:", len(customer_events))

Customer events: 96470


In [16]:
current_customer_orders = (
    customer_orders[
        [
            "order_id",
            "customer_unique_id",
            "order_purchase_timestamp"
        ]
    ]
    .sort_values(
        ["order_purchase_timestamp", "customer_unique_id"]
    )
    .reset_index(drop=True)
)

customer_history = pd.merge_asof(
    current_customer_orders,
    customer_events[
        [
            "customer_unique_id",
            "order_delivered_customer_date",
            "completed_orders_so_far",
            "completed_lates_so_far",
            "completed_delay_sum_so_far"
        ]
    ].sort_values(
        ["order_delivered_customer_date", "customer_unique_id"]
    ),
    left_on="order_purchase_timestamp",
    right_on="order_delivered_customer_date",
    by="customer_unique_id",
    direction="backward",
    allow_exact_matches=False
)

print("Customer history rows:", len(customer_history))
print(
    "Unique orders:",
    customer_history["order_id"].nunique()
)

Customer history rows: 96470
Unique orders: 96470


In [17]:
customer_history["customer_previous_orders"] = (
    customer_history["completed_orders_so_far"]
    .fillna(0)
)

customer_history["customer_previous_lates"] = (
    customer_history["completed_lates_so_far"]
    .fillna(0)
)

customer_history["customer_history_available"] = (
    customer_history["customer_previous_orders"] > 0
).astype(int)

customer_history["customer_historical_late_rate"] = (
    customer_history["completed_lates_so_far"]
    / customer_history["completed_orders_so_far"]
)

customer_history["customer_historical_avg_delay"] = (
    customer_history["completed_delay_sum_so_far"]
    / customer_history["completed_orders_so_far"]
)

customer_history_features = (
    customer_history
    .groupby("order_id")
    .agg(
        customer_previous_orders=(
            "customer_previous_orders",
            "mean"
        ),
        customer_historical_late_rate=(
            "customer_historical_late_rate",
            "mean"
        ),
        customer_historical_avg_delay=(
            "customer_historical_avg_delay",
            "mean"
        ),
        customer_history_available=(
            "customer_history_available",
            "max"
        )
    )
    .reset_index()
)

print("Rows:", len(customer_history_features))
print(
    "Unique orders:",
    customer_history_features["order_id"].nunique()
)

Rows: 96470
Unique orders: 96470


In [18]:
print(
    "Orders with customer history:",
    customer_history_features[
        "customer_history_available"
    ].sum()
)

print(
    "Orders without customer history:",
    (
        customer_history_features[
            "customer_history_available"
        ] == 0
    ).sum()
)

Orders with customer history: 2011
Orders without customer history: 94459


In [19]:
historical_features = (
    seller_history_features
    .merge(
        customer_history_features,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

print("Historical feature rows:", len(historical_features))
print(
    "Unique orders:",
    historical_features["order_id"].nunique()
)

Historical feature rows: 96470
Unique orders: 96470


In [20]:
historical_features.head()

,order_id,seller_previous_orders,seller_historical_late_rate,seller_historical_avg_delay,seller_history_available,customer_previous_orders,customer_historical_late_rate,customer_historical_avg_delay,customer_history_available
0,00010242fe8c5a6d1ba2dd792cb16214,74.0,0.013514,-12.072272,1,0.0,NaN,NaN,0
1,00018f77f2f0320c557190d7a144bdd3,0.0,NaN,NaN,0,0.0,NaN,NaN,0
2,000229ec398224ef6ca0657da4fc703e,4.0,0.000000,-13.860472,1,0.0,NaN,NaN,0
3,00024acbcdf0a6daa1e931b038114c75,10.0,0.100000,-13.687788,1,0.0,NaN,NaN,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,2.0,0.000000,-33.279583,1,0.0,NaN,NaN,0


In [21]:
history_numeric = [
    "seller_previous_orders",
    "seller_historical_late_rate",
    "seller_historical_avg_delay",
    "customer_previous_orders",
    "customer_historical_late_rate",
    "customer_historical_avg_delay"
]

for col in history_numeric:
    historical_features[col] = (
        historical_features[col]
        .fillna(0)
    )

In [24]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")

baseline_path = DATA_DIR / "processed" / "baseline_data.csv"

print("Looking for:", baseline_path.resolve())
print("Exists:", baseline_path.exists())

baseline_data = pd.read_csv(baseline_path)

print("Baseline rows:", len(baseline_data))
print("Baseline columns:", len(baseline_data.columns))

Looking for: C:\Projects\delivery-risk-engine\data\processed\baseline_data.csv
Exists: True
Baseline rows: 96470
Baseline columns: 28


In [25]:
historical_data = baseline_data.merge(
    historical_features,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

print("Historical dataset rows:", len(historical_data))
print(
    "Unique orders:",
    historical_data["order_id"].nunique()
)

Historical dataset rows: 96470
Unique orders: 96470


In [26]:
print(
    "Seller history coverage:",
    historical_data["seller_history_available"].mean()
)

print(
    "Customer history coverage:",
    historical_data["customer_history_available"].mean()
)

Seller history coverage: 0.946366746138696
Customer history coverage: 0.020845858816212294


In [27]:
historical_data[
    [
        "seller_previous_orders",
        "seller_historical_late_rate",
        "seller_historical_avg_delay",
        "customer_previous_orders",
        "customer_historical_late_rate",
        "customer_historical_avg_delay"
    ]
].isna().sum()

seller_previous_orders           0
seller_historical_late_rate      0
seller_historical_avg_delay      0
customer_previous_orders         0
customer_historical_late_rate    0
customer_historical_avg_delay    0
dtype: int64

In [28]:
history_features = [
    "seller_previous_orders",
    "seller_historical_late_rate",
    "seller_historical_avg_delay",
    "seller_history_available",
    "customer_previous_orders",
    "customer_historical_late_rate",
    "customer_historical_avg_delay",
    "customer_history_available",
]

historical_data = (
    historical_data
    .sort_values("order_purchase_timestamp")
    .reset_index(drop=True)
)

n = len(historical_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

hist_train = historical_data.iloc[:train_end].copy()
hist_val = historical_data.iloc[train_end:val_end].copy()
hist_test = historical_data.iloc[val_end:].copy()

print("Train:", hist_train.shape)
print("Validation:", hist_val.shape)
print("Test:", hist_test.shape)

Train: (67529, 36)
Validation: (14470, 36)
Test: (14471, 36)


In [29]:
DROP_COLUMNS = [
    "late_delivery",
    "order_id",
    "customer_id",
    "order_purchase_timestamp"
]

X_hist_train = hist_train.drop(columns=DROP_COLUMNS)
y_hist_train = hist_train["late_delivery"]

X_hist_val = hist_val.drop(columns=DROP_COLUMNS)
y_hist_val = hist_val["late_delivery"]

X_hist_test = hist_test.drop(columns=DROP_COLUMNS)
y_hist_test = hist_test["late_delivery"]

print("X:", X_hist_train.shape)
print("Target:", y_hist_train.shape)

X: (67529, 32)
Target: (67529,)


In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

numeric_hist = X_hist_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_hist = X_hist_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_pipeline_hist = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline_hist = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

hist_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline_hist, numeric_hist),
    ("categorical", categorical_pipeline_hist, categorical_hist)
])

historical_model = Pipeline([
    ("preprocessor", hist_preprocessor),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

historical_model.fit(
    X_hist_train,
    y_hist_train
)

print("Historical-intelligence model trained.")

C:\Users\anant\AppData\Local\Temp\ipykernel_20548\931106837.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_hist = X_hist_train.select_dtypes(


Historical-intelligence model trained.


In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

hist_pred = historical_model.predict(X_hist_val)
hist_prob = historical_model.predict_proba(X_hist_val)[:, 1]

historical_metrics = {
    "accuracy": accuracy_score(y_hist_val, hist_pred),
    "precision": precision_score(
        y_hist_val, hist_pred, zero_division=0
    ),
    "recall": recall_score(
        y_hist_val, hist_pred, zero_division=0
    ),
    "f1": f1_score(
        y_hist_val, hist_pred, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_hist_val, hist_prob
    ),
    "pr_auc": average_precision_score(
        y_hist_val, hist_prob
    )
}

historical_metrics

{'accuracy': 0.5783690393918453,
 'precision': 0.09340659340659341,
 'recall': 0.7917205692108668,
 'f1': 0.167098976109215,
 'roc_auc': 0.7526487372566546,
 'pr_auc': 0.13973334236554052}

In [32]:
print(confusion_matrix(y_hist_val, hist_pred))

[[7757 5940]
 [ 161  612]]


In [33]:
comparison = pd.DataFrame([
    {
        "model": "Enhanced Baseline",
        "accuracy": 0.7009397457158651,
        "precision": 0.11198428290766209,
        "recall": 0.6636481241914618,
        "f1": 0.19163242435562197,
        "roc_auc": 0.765886066224983,
        "pr_auc": 0.1505465350766382
    },
    {
        "model": "Baseline + Historical Intelligence",
        **historical_metrics
    }
])

comparison

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Enhanced Baseline,0.700940,0.111984,0.663648,0.191632,0.765886,0.150547
1,Baseline + Historical Intelligence,0.578369,0.093407,0.791721,0.167099,0.752649,0.139733
